# v0.38.0 — Migration system

## 1. Intro

Since v0.38.0 the ORM writes Django-style migration files from the models and applies them, with the full
SurrealDB-ORM's API (`surreal_orm_lite.migrations`):

- `makemigrations(name, models=…)` compares the models with the connected database — with the migration files not yet
  applied replayed over it — and writes the next file of the migrations directory: plain Python, a `Migration` and
  its operations (`CreateTable`, `AddField`, `CreateIndex`, `DefineEvent` …), to review before applying;
- `MigrationExecutor(dir).migrate()` applies each pending migration in one SurrealDB transaction together with its
  record in the `_surreal_orm_migrations` table, so a migration is applied entirely or not at all;
- `rollback(target)` undoes the migrations applied after *target*, `upgrade()` runs the data operations
  (`DataMigration`), `get_migration_status()` and `show_sql()` describe the files;
- `schema_diff(models=…)` lists what would bring the database to the models — `[]` when they agree.

The same migrations apply on SurrealDB 3.x, 2.x and the embedded engine, except the forms the README's section 39
lists (a full-text index is written `FULLTEXT` on 3.x and `SEARCH` on 2.x, for instance). The notebook runs top to
bottom on both lines; the outputs below come from a SurrealDB 3.x server — on 2.x, §6 prints `SEARCH` where 3.x prints
`FULLTEXT`.

## 2. Connect

`SURREALDB_HOST` and `SURREALDB_PORT` choose the server (default `localhost:8000`), over WebSocket. The notebook works
in a database of its own, `migrations_demo`, removed first so that it can be re-executed in place.

In [1]:
import os

from surreal_orm_lite import SurrealDBConnectionManager

HOST = os.environ.get("SURREALDB_HOST", "localhost")
PORT = os.environ.get("SURREALDB_PORT", "8000")
# WebSocket (.../rpc), as the other notebooks.
SurrealDBConnectionManager.set_connection(
    url=f"ws://{HOST}:{PORT}/rpc",
    user="root",
    password="root",
    namespace="examples",
    database="migrations_demo",
)
client = await SurrealDBConnectionManager.get_client()
await client.query("REMOVE DATABASE IF EXISTS migrations_demo; DEFINE DATABASE migrations_demo;", {})
print("Connection configured:", SurrealDBConnectionManager.is_connection_set())

Connection configured: True


## 3. The models, version 1

`Author`, `Post` and `Audit` are `SCHEMAFULL` tables. `Post` declares an index on `title` and an event, `post_audit`,
that logs each new post in the table `audit`, which `Audit` models — a table no model declares would be a removal for
`makemigrations()` (left out with a warning, unless `drop_missing=True`). The migration files go to a temporary directory; an application keeps them in
its repository (`migrations/` by default). `show()` prints a file without its `Generated:` header line, the time it
was written.

In [2]:
import shutil
import tempfile
from pathlib import Path

from surreal_orm_lite import BaseSurrealModel, Event, Index, SchemaMode, SurrealConfigDict


class Author(BaseSurrealModel):
    model_config = SurrealConfigDict(table_name="author", schema_mode=SchemaMode.SCHEMAFULL)

    id: str | None = None
    name: str


class Post(BaseSurrealModel):
    model_config = SurrealConfigDict(
        table_name="post",
        schema_mode=SchemaMode.SCHEMAFULL,
        indexes=[Index("post_title", "title")],
        events=[
            Event(
                "post_audit",
                when="$event = 'CREATE'",
                then="CREATE audit SET title = $after.title, action = $event",
            )
        ],
    )

    id: str | None = None
    title: str
    author: str


class Audit(BaseSurrealModel):
    model_config = SurrealConfigDict(table_name="audit", schema_mode=SchemaMode.SCHEMAFULL)

    id: str | None = None
    title: str
    action: str


MIGRATIONS_DIR = Path(tempfile.mkdtemp())


def show(path: Path) -> None:
    """Print a migration file, without the line of the time it was generated."""
    print(f"# {path.name}")
    print("\n".join(line for line in path.read_text().splitlines() if not line.startswith("Generated:")))

## 4. `makemigrations()`: the first migration

`makemigrations()` reads the connected database — empty here — by default (`from_db=True`, the full ORM CLI's
`--from-db`) and writes `0001_initial.py`: a `CreateTable` and its `AddField` operations for each table, the index and
the event.

Run again before `migrate()`, it writes nothing and returns `None`: the pending `0001_initial` is replayed over the
database, so a new migration never repeats a pending one.

In [3]:
from surreal_orm_lite.migrations import makemigrations

path = await makemigrations("initial", migrations_dir=MIGRATIONS_DIR, models=[Author, Post, Audit])
show(path)

# 0001_initial.py
"""
Migration: 0001_initial

Auto-generated migration file. Review before applying.
"""

from surreal_orm_lite.migrations import Migration
from surreal_orm_lite.migrations.operations import (
    AddField,
    CreateIndex,
    CreateTable,
    DefineEvent,
)


migration = Migration(
    name="0001_initial",
    dependencies=[],
    operations=[
        CreateTable(name='audit', table_type='normal'),
        AddField(table='audit', name='action', field_type='string'),
        AddField(table='audit', name='title', field_type='string'),
        CreateTable(name='author', table_type='normal'),
        AddField(table='author', name='name', field_type='string'),
        CreateTable(name='post', table_type='normal'),
        AddField(table='post', name='author', field_type='string'),
        AddField(table='post', name='title', field_type='string'),
        CreateIndex(table='post', name='post_title', fields=['title']),
        DefineEvent(
            name='post_audit',
   

In [4]:
print(await makemigrations("again", migrations_dir=MIGRATIONS_DIR, models=[Author, Post, Audit]))

None


## 5. `migrate()`

`MigrationExecutor(directory).migrate()` checks the pending migrations against the database's schema, then applies
each one in a transaction with its record, and returns the names it applied. `get_migration_status()` describes every
file. The tables are then ordinary ORM tables.

In [5]:
from surreal_orm_lite.migrations import MigrationExecutor

executor = MigrationExecutor(MIGRATIONS_DIR)
print(await executor.migrate())
for name, status in (await executor.get_migration_status()).items():
    print(name, status)

await Author(id="ada", name="Ada").save()
await Post(id="p1", title="Migrations in SurrealDB", author="ada").save()
await Post(id="p2", title="Schema drift", author="ada").save()
print("posts:", await Post.objects().count(), "| audit rows:", await Audit.objects().count())

['0001_initial']
0001_initial {'applied': True, 'reversible': True, 'has_data': False, 'operations': 10}
posts: 2 | audit rows: 2


## 6. The models, version 2

`Post` gains a `rating: int = 0` field, a full-text index on `title` with the analyzer `words`, and its event now
logs updates too. `makemigrations("add_rating")` writes `0002_add_rating.py`; each operation's `describe()` says what
it does. `show_sql()` prints a migration's statements without a request, for a server line: here the connected
server's, read by `cached_major()` (3 from SurrealDB 3 on, else 2) — the full-text index is `FULLTEXT` on 3.x and
`SEARCH` on 2.x. The event's redefinition carries its first definition (`previous=`), so the migration can be rolled
back (§10).

In [6]:
from surreal_orm_lite import Analyzer
from surreal_orm_lite.migrations import Migration
from surreal_orm_lite.time_travel import cached_major

WORDS = Analyzer("words", ["blank"], ["lowercase"])


class Post(BaseSurrealModel):  # version 2
    model_config = SurrealConfigDict(
        table_name="post",
        schema_mode=SchemaMode.SCHEMAFULL,
        indexes=[Index("post_title", "title"), Index("post_title_search", "title", search_analyzer=WORDS)],
        events=[
            Event(
                "post_audit",
                when="$event = 'CREATE' OR $event = 'UPDATE'",
                then="CREATE audit SET title = $after.title, action = $event",
            )
        ],
    )

    id: str | None = None
    title: str
    author: str
    rating: int = 0


path = await makemigrations("add_rating", migrations_dir=MIGRATIONS_DIR, models=[Author, Post, Audit])
for operation in executor.load_migration(path.stem).operations:
    print("-", operation.describe())

- Define analyzer words
- Add field rating to post
- Create index post_title_search on post
- Define event post_audit on post


In [7]:
line = 3 if await cached_major(client) >= 3 else 2
print(f"-- SurrealDB {line}.x")
print(await executor.show_sql("0002_add_rating", line=line))

-- SurrealDB 3.x
DEFINE ANALYZER OVERWRITE `words` TOKENIZERS blank FILTERS lowercase;

DEFINE FIELD `rating` ON `post` TYPE int DEFAULT 0;

DEFINE INDEX `post_title_search` ON `post` FIELDS `title` FULLTEXT ANALYZER `words`;

DEFINE EVENT OVERWRITE `post_audit` ON `post` WHEN $event = 'CREATE' OR $event = 'UPDATE' THEN { CREATE audit SET title = $after.title, action = $event };


In [8]:
print(await executor.migrate())

['0002_add_rating']


## 7. Atomicity

A hand-written migration creates a table `draft`, then runs a `RawSQL` that fails (`THROW`). `migrate()` raises
`MigrationStatementError`, which names the migration, the operation and SurrealDB's reason — never the statement. The
migration's transaction was cancelled: the table `draft` does not exist, and the migration is not recorded. The file
is then removed.

In [9]:
from surreal_orm_lite.migrations import MigrationStatementError

broken = MIGRATIONS_DIR / "0003_broken.py"
broken.write_text(
    """
from surreal_orm_lite.migrations import CreateTable, Migration, RawSQL

migration = Migration(
    name="0003_broken",
    dependencies=["0002_add_rating"],
    operations=[
        CreateTable(name="draft"),
        RawSQL("THROW 'not today';"),
    ],
)
"""
)
try:
    await executor.migrate()
except MigrationStatementError as error:
    print("MigrationStatementError:", error)

info = (await client.query("INFO FOR DB;", {}))
print("tables:", sorted(info["tables"]))
print("recorded:", [name for name, status in (await executor.get_migration_status()).items() if status["applied"]])
broken.unlink()

MigrationStatementError: Migration '0003_broken' failed at operation 2 of 2 (Raw SQL): An error occurred: not today
tables: ['_surreal_orm_migrations', 'audit', 'author', 'post']


recorded: ['0001_initial', '0002_add_rating']


## 8. A data migration

`DataMigration` transforms records. `migrate()` applies the schema only (`schema_only=True`, the default), so the
migration below is recorded without touching a post; `upgrade()` then runs the data operations of the applied
migrations. Data operations are not recorded — `upgrade()` runs them again each time — so they are written
idempotent. Its `backwards_sql` makes it reversible.

In [10]:
(MIGRATIONS_DIR / "0003_rate_posts.py").write_text(
    """
from surreal_orm_lite.migrations import DataMigration, Migration

migration = Migration(
    name="0003_rate_posts",
    dependencies=["0002_add_rating"],
    operations=[
        DataMigration(
            forwards_sql="UPDATE post SET rating = 5;",
            backwards_sql="UPDATE post SET rating = 0;",
        ),
    ],
)
"""
)
print("migrate():", await executor.migrate())
print("before upgrade():", [(post.id, post.rating) for post in await Post.objects().order_by("id").exec()])
print("upgrade():", await executor.upgrade())
print("after upgrade():", [(post.id, post.rating) for post in await Post.objects().order_by("id").exec()])

migrate(): ['0003_rate_posts']
before upgrade(): [('p1', 0), ('p2', 0)]
upgrade(): ['0003_rate_posts']
after upgrade(): [('p1', 5), ('p2', 5)]


## 9. `schema_diff()`

The database now matches the models: `schema_diff()` returns `[]`. The comparison normalises what SurrealDB renders
differently from the ORM, so the diff right after `migrate()` is empty on every line.

In [11]:
from surreal_orm_lite import schema_diff

print(await schema_diff(models=[Author, Post, Audit]))

[]


## 10. `rollback()`

`rollback("0001_initial")` undoes the migrations applied after `0001_initial`, the last first, each in a transaction
that also deletes its record: `0003_rate_posts` runs its `backwards_sql`, `0002_add_rating` removes the field and the
full-text index and restores the event's first definition.

In [12]:
print(await executor.rollback("0001_initial"))
for name, status in (await executor.get_migration_status()).items():
    print(name, status["applied"])

post_info = await client.query("INFO FOR TABLE post;", {})
print("fields:", sorted(post_info["fields"]))
print("indexes:", sorted(post_info["indexes"]))
print("post_audit:", post_info["events"]["post_audit"])

['0003_rate_posts', '0002_add_rating']
0001_initial True
0002_add_rating False
0003_rate_posts False
fields: ['author', 'title']
indexes: ['post_title']
post_audit: DEFINE EVENT post_audit ON post WHEN $event = 'CREATE' THEN { CREATE audit SET title = $after.title, action = $event }


## 11. Cleanup

The demo database, the connection and the migrations directory are removed.

In [13]:
await client.query("REMOVE DATABASE IF EXISTS migrations_demo;", {})
await SurrealDBConnectionManager.close_connection()
shutil.rmtree(MIGRATIONS_DIR)
print("Cleaned up.")

Cleaned up.
